# WasteLens - Colab Training (MobileNetV2, transfer learning)

Same pipeline as `src/train.py` (kagglehub download -> 4-bin remap -> stratified 70/15/15 split -> frozen MobileNetV2 + head), then **trains** with class weights and saves checkpoints after every epoch.

## GPU runtime setup - do this FIRST

1. **Runtime -> Change runtime type -> Hardware accelerator: GPU (T4) -> Save.**
2. Run the environment-check cell below; it should list a GPU (`nvidia-smi`).
3. Cell 3 mounts **Google Drive** - epoch checkpoints, the training log and the final model are saved to `MyDrive/WasteLens/`, so a recycled runtime never loses progress. If Drive is unavailable, the notebook falls back to ephemeral `/content/wastelens_output` and zips everything for download at the end.

## What to expect

- Cells 2-5 reproduce the **pre-training summary** (the same stop point as `src/train.py`): dataset download -> remap -> per-bin split table -> model build -> class weights.
- Cell 6 runs `model.fit` (**class weights applied**, checkpoint after every epoch, CSV log). Roughly 1.5 min/epoch on a T4 (~15-20 min for 10 epochs); CPU-only is ~50-100x slower.
- Cell 7 evaluates on the held-out test split and plots curves (full per-class precision/recall/F1 comes from `src/evaluate.py`).
- Cell 8 **exports**: the final `.keras` model + `labels.json` (bin order, 224x224, `[-1,1]` preprocessing) + optional TensorFlow.js conversion into `models/tfjs_model/`.

**If the runtime is recycled mid-training:** rerun cells 2-5, then either retrain from scratch or load a checkpoint from `MyDrive/WasteLens/models/checkpoints/wastelens_epNN.keras` with `tf.keras.models.load_model` and continue.


In [ ]:
import sys
from packaging.version import Version

print("Python:", sys.version.split()[0])
gpu_line = !nvidia-smi -L 2>/dev/null || echo "NO GPU DETECTED - Runtime > Change runtime type > GPU (T4)"
print("GPU:", *gpu_line, sep="\n  ")

%pip install -q -U kagglehub

import tensorflow as tf
print("TensorFlow:", tf.__version__)
assert Version(tf.__version__) >= Version("2.19"), (
    "Need TF >= 2.19 (Keras 3) for tf.keras.applications.MobileNetV2")
print("TF-visible devices:", tf.config.list_physical_devices("GPU") or "none - CPU only (slow!)")


In [ ]:
from pathlib import Path

OUT_ROOT = Path("/content/wastelens_output")  # ephemeral fallback
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/WasteLens")
    DRIVE_MOUNTED = True
except Exception as exc:
    print("Google Drive not mounted - using ephemeral fallback:", exc)

CHECKPOINTS_DIR = OUT_ROOT / "models" / "checkpoints"
FINAL_DIR = OUT_ROOT / "models" / "final"
TFJS_DIR = OUT_ROOT / "models" / "tfjs_model"
for _d in (CHECKPOINTS_DIR, FINAL_DIR, TFJS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print("Outputs ->", OUT_ROOT, "(Drive, persistent)" if DRIVE_MOUNTED
      else "(EPHEMERAL - zipped for download in the final cell)")


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/WasteLens")
if not (REPO_DIR / "src" / "train.py").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Anik00223/WasteLens.git", str(REPO_DIR)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR / "src"))

import train as wl  # src/train.py - single source of truth for the pipeline

print("Imported pipeline from:", REPO_DIR / "src" / "train.py")
print("Bins (fixed label order):", wl.BINS)
print("Dropped classes:", sorted(wl.DROPPED_CLASSES))


In [ ]:
import math

dataset_root = wl.download_dataset()
images = wl.scan_images(dataset_root)
binned = wl.remap_to_bins(images)
splits = wl.stratified_split(binned)
wl.report_split_sizes(splits)

model, base = wl.build_model(num_bins=len(wl.BINS))
train_ds = wl.make_dataset(splits["train"], training=True)
val_ds = wl.make_dataset(splits["val"], training=False)
test_ds = wl.make_dataset(splits["test"], training=False)

for batch_images, batch_labels in train_ds.take(1):
    lo = float(batch_images.numpy().min())
    hi = float(batch_images.numpy().max())
    print("pipeline check - batch:", batch_images.shape, "labels:", batch_labels.shape,
          "range [%.2f, %.2f] (MobileNetV2 expects [-1, 1])" % (lo, hi))

class_weights = wl.compute_class_weights(splits["train"])
model.summary(line_length=110, show_trainable=True)

STEPS_PER_EPOCH = math.ceil(len(splits["train"]) / wl.BATCH_SIZE)
print("\nsteps_per_epoch:", STEPS_PER_EPOCH)
print("=== PRE-TRAINING SUMMARY ABOVE - same stop point as src/train.py ===")
print("Review the counts and split above; run the next cell to start model.fit.")


In [ ]:
import tensorflow as tf

EPOCHS = 10  # raise later if val accuracy is still climbing

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / "wastelens_ep{epoch:02d}.keras"),
        save_freq="epoch",
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(OUT_ROOT / "training_log.csv"), append=True),
]

history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

print("Last-epoch metrics:", {k: v[-1] for k, v in history.history.items()})
print("Epoch checkpoints ->", CHECKPOINTS_DIR)


In [ ]:
import matplotlib.pyplot as plt

loss, acc = model.evaluate(test_ds, verbose=0)
print("Held-out TEST loss: %.4f  accuracy: %.4f" % (loss, acc))
print("(per-class precision/recall/F1 comes later from src/evaluate.py)")

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h["loss"], label="train")
axes[0].plot(h["val_loss"], label="val")
axes[0].set_title("loss")
axes[0].legend()
axes[1].plot(h["sparse_categorical_accuracy"], label="train")
axes[1].plot(h["val_sparse_categorical_accuracy"], label="val")
axes[1].set_title("accuracy")
axes[1].legend()
plt.show()


In [ ]:
import json
import shutil
import subprocess

final_path = FINAL_DIR / "wastelens_mobilenetv2.keras"
model.save(final_path)
meta = {
    "bins": wl.BINS,
    "img_size": list(wl.IMG_SIZE),
    "preprocessing": "mobilenet_v2.preprocess_input -> range [-1, 1]",
    "epochs_trained": len(history.history["loss"]),
    "dataset_handle": wl.KAGGLE_HANDLE,
}
(FINAL_DIR / "labels.json").write_text(json.dumps(meta, indent=2))
print("Saved Keras model  ->", final_path)
print("Saved labels/meta  ->", FINAL_DIR / "labels.json")

try:
    %pip install -q tensorflowjs
    subprocess.run(
        ["tensorflowjs_converter", "--input_format=keras_saved_model",
         str(final_path), str(TFJS_DIR)],
        check=True,
    )
    print("TF.js model written ->", TFJS_DIR)
except Exception as exc:
    print("TF.js conversion skipped (the .keras file above is the source of truth):", exc)

if not DRIVE_MOUNTED:
    zip_path = shutil.make_archive("/content/wastelens_output", "zip", str(OUT_ROOT))
    from google.colab import files
    files.download(zip_path)
    print("Ephemeral fallback: zipped outputs and triggered download ->", zip_path)
